# Lab 2: a prompt test bench

Score prompt versions over 30 labelled cases, save each run and compare two runs case by
case. This notebook follows [Lab 2 in the book](https://handbook.aggarwal.dev/lab-prompt-bench.html): read each
step there, then run it here. The code is in `labs/02-prompt-bench/bench.py`.

On the fake backend, keyword rules stand in for the model, so the numbers test the
bench's plumbing, not a prompt. Run on Claude or Ollama to compare prompts.

## Set up

Run the next cell first. It finds the book's code (in Colab it downloads it), installs what
the labs need and picks who answers:

- `fake` needs no key and costs nothing. Its replies are scripted, so they show how the
  code works, not what a model would say.
- `anthropic` uses Claude with your own API key. In Colab, add `ANTHROPIC_API_KEY` in the
  Secrets panel (the key icon on the left) and let this notebook read it. Elsewhere the cell
  asks for the key. It is never written into the notebook.
- `ollama` runs a small open model, free. In Colab it installs and starts Ollama for you:
  switch to a GPU first (Runtime, then Change runtime type, then T4 GPU), and expect the
  first run to download about 2 GB.

To change the backend later, change it here and run this cell again. Lab 0 in the book
explains the choices: https://handbook.aggarwal.dev/lab-setup.html

In [ ]:
# @title Set up this lab
BACKEND = "fake"  # @param ["fake", "anthropic", "ollama"]
MODEL = ""  # @param {type:"string"}

import os, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

REPO = "https://github.com/aggarwaldev/build-ai-handbook.git"


def find_root():
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "labs" / "common" / "llm.py").is_file():
            return folder
    return None


root = find_root()
if root is None:
    if not Path("build-ai-handbook").is_dir():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPO], check=True)
    root = Path("build-ai-handbook").resolve()
os.chdir(root)


def run_command(*cmd):
    """Run a command from the repository root, show its output in this cell, and stop if it fails."""
    done = subprocess.run([sys.executable if part == "python" else part for part in cmd],
                          capture_output=True, text=True)
    print(done.stdout, end="")
    print(done.stderr, end="", file=sys.stderr)
    if done.returncode:
        raise RuntimeError("this command failed: " + " ".join(cmd))


if str(root / "labs") not in sys.path:
    sys.path.insert(0, str(root / "labs"))

try:
    import anthropic, numpy  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "labs/requirements.txt"], check=True)

os.environ["LLM_BACKEND"] = BACKEND
if MODEL.strip():
    os.environ["LLM_MODEL"] = MODEL.strip()
else:
    os.environ.pop("LLM_MODEL", None)
IN_COLAB = "google.colab" in sys.modules

if BACKEND == "anthropic" and not os.environ.get("ANTHROPIC_API_KEY", "").strip():
    if IN_COLAB:
        from google.colab import userdata
        try:
            os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        except Exception as err:
            raise RuntimeError("Add ANTHROPIC_API_KEY in the Secrets panel (the key icon on the left) "
                               "and allow this notebook to read it, then run this cell again.") from err
    else:
        from getpass import getpass
        os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key (not saved in the notebook): ")


def ollama_running():
    host = os.environ.get("OLLAMA_HOST", "http://localhost:11434").rstrip("/")
    try:
        urllib.request.urlopen(host + "/api/tags", timeout=2)
        return True
    except OSError:
        return False


if BACKEND == "ollama":
    if not ollama_running():
        if not IN_COLAB:
            raise RuntimeError("Start Ollama on this machine (ollama serve), or pick another backend.")
        if shutil.which("ollama") is None:
            if shutil.which("zstd") is None:
                subprocess.run("apt-get -qq update && apt-get -qq install -y zstd", shell=True, check=True)
            subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
        subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "a"), stderr=subprocess.STDOUT)
        for _ in range(60):
            if ollama_running():
                break
            time.sleep(1)
        else:
            raise RuntimeError("Ollama did not start; see ollama.log.")
    if shutil.which("nvidia-smi") is None:
        print("No GPU found: Ollama will answer slowly on the CPU.")
    pulled = os.environ.get("LLM_MODEL", "llama3.2")
    if shutil.which("ollama"):
        subprocess.run(["ollama", "pull", pulled], check=True)

print(f"Ready: code in {root}, backend {BACKEND}" + (f", model {os.environ['LLM_MODEL']}" if os.environ.get("LLM_MODEL") else ""))

## Step 1: cases with a right answer

Each line of `cases.jsonl` is a message and its correct category. Look at the first few,
and at how many cases each category has.

In [ ]:
sys.path.insert(0, str(root / "labs" / "02-prompt-bench"))
import bench
from collections import Counter

cases = bench.load_cases()
for case in cases[:3]:
    print(case)
print(len(cases), "cases:", dict(Counter(c["label"] for c in cases)))

## Step 2: prompt versions in files

Each version is a text file sent as the system prompt; the message is wrapped the same
way every time. Print v1, then v2, and see what v2 adds.

In [ ]:
for version in ("v1", "v2"):
    name, prompt = bench.load_prompt(version)
    print(f"--- {name} (sha {bench.fingerprint(prompt)}) ---")
    print(prompt)
print("--- the user turn ---")
print(bench.user_turn("Can I book my bike in?"))

## Step 3: score each reply

`parse_reply` reads a reply as `(category, json_ok)`, so format and accuracy are scored
apart. Try a clean reply, a reply in a code fence and a bare answer.

In [ ]:
for text in ('{"category": "booking"}',
             'Sure! ```json\n{"category": "quote"}\n```',
             "Category: parts.",
             "I think it is an appointment"):
    print(repr(text), "->", bench.parse_reply(text))

## Step 4: the report

`run` sends every case, scores every reply and prints a report. On the fake backend this
is the page's v2 report: 22 of 30 right, quote the worst category.

In [ ]:
!python labs/02-prompt-bench/bench.py run v2

`-v` shows every case as it is scored, and `--limit` tries a version on the first few
cases. `--no-save` keeps the run out of `runs/`.

In [ ]:
!python labs/02-prompt-bench/bench.py run v2 --limit 6 --no-save -v

## Step 5: the fake model, and what its numbers mean

The fake's keyword rules are in `bench.KEYWORD_RULES`, and the first rule with a matching
word wins. It cannot read definitions or examples, which is why v2 and v3 will score the
same in the next step.

In [ ]:
for label, words in bench.KEYWORD_RULES:
    print(f"{label:<10}", ", ".join(words))

## Step 6: save runs and compare them case by case

A real model would cost money here: one run is 30 calls, about half a cent on
`claude-haiku-5-5` (the page works it out), and the fake backend and Ollama cost nothing.
Run all three versions on the fake backend, then list them. The cell hides their reports,
which you have seen.

In [ ]:
!python labs/02-prompt-bench/bench.py run v1 > /dev/null 2>&1
!python labs/02-prompt-bench/bench.py run v2 > /dev/null 2>&1
!python labs/02-prompt-bench/bench.py run v3 > /dev/null 2>&1
!python labs/02-prompt-bench/bench.py list

`compare` needs two run names. This cell picks the newest run of each version from the
saved runs, then compares v1 with v2: v2 fixes cases and breaks none.

In [ ]:
def latest(version):
    runs = [bench.load_run(p.stem) for p in sorted(bench.RUNS.glob("*.json"))]
    return [r["id"] for r in runs if r["prompt"] == version][-1]

run_v1, run_v2, run_v3 = latest("v1"), latest("v2"), latest("v3")
run_command("python", "labs/02-prompt-bench/bench.py", "compare", run_v1, run_v2)

v2 against v3 changes nothing on the fake, as step 5 predicts. v3 does cost more input
tokens per case, for no gain here.

In [ ]:
run_command("python", "labs/02-prompt-bench/bench.py", "compare", run_v2, run_v3)

## Step 7: why thirty cases is a small number

`wilson` gives the 95% interval for a score. 22 of 30 is a band about 30 points wide, and
23 of 30 is almost the same band, so one more right answer proves nothing. The width
shrinks only with the square root of the number of cases.

In [ ]:
for k, n in ((22, 30), (23, 30), (75, 100), (225, 300), (900, 1200)):
    lo, hi = bench.wilson(k, n)
    print(f"{k:>4}/{n:<5} {lo:.3f} to {hi:.3f}   width {hi - lo:.3f}")

The sign test looks only at the cases that changed. Five fixed and none broken is
still p = 0.0625, the worked example in chapter 26.

In [ ]:
print("18 fixed, 0 broken:", bench.sign_test(18, 0))
print(" 5 fixed, 0 broken:", bench.sign_test(5, 0))

## Run the tests

In [ ]:
!cd labs && python -m pytest -q -p no:cacheprovider 02-prompt-bench

## Experiments

Try the [experiments on the lab page](https://handbook.aggarwal.dev/lab-prompt-bench.html#experiments). A quick one
is to break the format on purpose: this copy of v2 loses its JSON instruction, so accuracy
stays while JSON validity falls to nothing. The cell removes the copy afterwards.

In [ ]:
_, v2_text = bench.load_prompt("v2")
lines = v2_text.rstrip("\n").split("\n")
nojson = bench.PROMPTS / "v2-nojson.txt"
nojson.write_text("\n".join(lines[:-2]) + "\n", encoding="utf-8")
try:
    run_command("python", "labs/02-prompt-bench/bench.py", "run", "v2-nojson", "--no-save")
finally:
    nojson.unlink()

Edit the cells above to try others, such as your own cases with `--cases`. Then tick the
lab's checklist in the book: your progress is kept in your browser, not in this notebook.